In [1]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../..")

from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_beus_articulatory_features

In [5]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_beus.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])

In [6]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,ac_f,art_f,subject
0,1.47,1.53,1.53,1.60,p,l,p,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
1,1.53,1.60,1.60,1.75,l,i,p_l,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
2,1.60,1.75,1.75,1.83,i,z,p_l_i,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
3,1.83,1.97,1.97,2.16,k,ɔ,k,call,1,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
4,1.97,2.16,2.16,2.17,ɔ,l,k_ɔ,call,1,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
...,...,...,...,...,...,...,...,...,...,...,...,...
163314,2.23,2.32,2.32,2.39,h,i,h,he,6,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
163315,2.39,2.53,2.53,2.67,f,ɛ,f,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
163316,2.53,2.67,2.67,2.73,ɛ,l,f_ɛ,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
163317,2.67,2.73,2.73,2.79,l,i,f_ɛ_l,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me


In [ ]:
tokens = {
    "start": [],
    "end": [],
    "syllable": [],
    "art_f": [],
    "ac_f": [],
    "subject": [],
}

metadata = set(
    [(start, end, target, art_f, ac_f, subject) for start, end, target, art_f, ac_f, subject in zip(df["syllable1_start"], df["syllable1_end"], df["syllable1"], df["art_f"], df["ac_f"], df["subject"])] + 
    [(start, end, target, art_f, ac_f, subject) for start, end, target, art_f, ac_f, subject in zip(df["syllable2_start"], df["syllable2_end"], df["syllable2"], df["art_f"], df["ac_f"], df["subject"])]
)

metadata = sorted(list(metadata), key=lambda x: (x[3], x[4], x[5]))

for start, end, target, art_f, ac_f, subject in metadata:
    tokens["start"].append(start)
    tokens["end"].append(end)
    tokens["syllable"].append(target)
    tokens["art_f"].append(art_f)
    tokens["ac_f"].append(ac_f)
    tokens["subject"].append(subject)

In [8]:
token_df = pd.DataFrame(tokens)

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from utils.models import VideoAutoencoder, SequenceAutoencoder

In [10]:
def latent_stats(z):
    # z: (B, latent_dim)
    return {
        "latent_var_mean": z.var(dim=0).mean().item(),
        "latent_var_min":  z.var(dim=0).min().item(),
        "latent_var_max":  z.var(dim=0).max().item(),
    }

def batch_diversity(x):
    # x: (B, ...)
    x_flat = x.reshape(x.size(0), -1)
    return x_flat.std(dim=0).mean().item()

def visualize(clean, recon, name="Reconstruction"):
    import matplotlib.pyplot as plt

    try:
        # Get both channels, frame 0 of first batch element
        clean_mri= clean[0, 0, 0].cpu().numpy()
    except RuntimeError:
        clean_mri = clean[0, 0, 0].detach().cpu().numpy()

    recon_mri = recon[0, 0, 0].detach().cpu().numpy()

    plt.figure(figsize=(10, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(clean_mri, cmap='gray')
    plt.title("Original (MRI)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(recon_mri, cmap='gray')
    plt.title(f"{name} (MRI)")
    plt.axis('off')

    plt.tight_layout()
    plt.savefig(f"video_visualization/beus/{name.lower()}.png", dpi=300)
    plt.close("all")

def combined_loss(y_true, y_pred, warped_x, criterion, plot=False, ratio=1):
    diff_true = y_true[:, :, 1:, :, :] - y_true[:, :, :-1, :, :]
    diff_pred = y_pred[:, :, 1:, :, :] - y_pred[:, :, :-1, :, :]
    loss = criterion(diff_true, diff_pred) * (1 - ratio) + criterion(y_true, y_pred) * ratio

    if plot:
        visualize(diff_true, diff_pred, name="Reconstruction (differential)")
        visualize(y_true, y_pred, name="Reconstruction")
        visualize(y_true, warped_x, name="Transformation")

    return loss

In [11]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

BATCH_SIZE = 4

In [11]:
log = {
    "batch_idx": [],
    "loss": [],
}

art_model = VideoAutoencoder(
    input_shape=(10, 56, 112),
    input_channels=2,
    latent_dim=1024,
    use_mlp=True,
    stn_mode="separate",
    grid_sample_cpu_fallback=True,
).to(device)

art_criterion = nn.MSELoss()
art_optimizer = torch.optim.Adam(art_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

art_model.train()

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    articulatory_features = get_beus_articulatory_features(art_f, start, end)

    if articulatory_features is None:
        continue

    batch.append(
        torch.tensor(articulatory_features, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1
    try:
        art_batch = torch.stack(batch).to(device)  # (B,T,P,C)
    except RuntimeError:
        continue
    batch.clear()

    # ===== Articulatory =====
    art_out, art_warped_x, art_z = art_model(art_batch, return_warped_x=True, return_latent=True)
    art_loss = combined_loss(art_batch, art_out, art_warped_x, art_criterion, plot=batch_idx % 30 == 0, ratio=0.5)

    art_optimizer.zero_grad()
    art_loss.backward()
    art_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(art_loss.item())
    pbar.set_description(
        f"{batch_idx}: ART {art_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if (batch_idx) % 300 == 0:
        with torch.no_grad():
            art_stats = latent_stats(art_z)

        print(
            f"[ART] latent var mean={art_stats['latent_var_mean']:.4e}, "
            f"min={art_stats['latent_var_min']:.4e}"
        )

        print(
            f"[ART] input std={batch_diversity(art_batch):.4e}, "
            f"recon std={batch_diversity(art_out):.4e}"
        )

        torch.save(art_model.state_dict(), "ckpts/beus/art_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/beus/art_model_training_log.csv", index=False)

torch.save(art_model.state_dict(), "ckpts/beus/art_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/beus/art_model_training_log.csv", index=False)

300: ART 0.3905:   1%|          | 1199/226488 [08:18<16:03:23,  3.90it/s]

[ART] latent var mean=1.1490e+03, min=2.2714e-01
[ART] input std=4.7577e-01, recon std=9.0867e-03


600: ART 0.4043:   1%|          | 2399/226488 [16:59<19:51:39,  3.13it/s]

[ART] latent var mean=1.1423e+04, min=1.9469e-01
[ART] input std=4.9491e-01, recon std=1.9427e-02


900: ART 0.3637:   2%|▏         | 3599/226488 [25:25<29:26:39,  2.10it/s]

...

56100: ART 0.3642:  99%|█████████▉| 224399/226488 [27:25:47<12:20,  2.82it/s]

[ART] latent var mean=4.0007e+08, min=2.9484e+02
[ART] input std=4.8911e-01, recon std=9.4843e-02


56400: ART 0.3841: 100%|█████████▉| 225599/226488 [27:34:38<05:29,  2.70it/s]

[ART] latent var mean=2.6747e+07, min=3.5620e+02
[ART] input std=5.1848e-01, recon std=7.1027e-02


56622: ART 0.3694: 100%|██████████| 226488/226488 [27:41:46<00:00,  2.27it/s]


In [14]:
def temporal_diff(x):
    """
    x: (B, T, ...)
    returns: (B, T-1, ...)
    """
    return x[:, 1:] - x[:, :-1]

def combined_loss(x_hat, x, alpha=0.5):
    """
    alpha: weight on static loss (delta weight = 1-alpha)
    """
    # static loss
    loss_static = F.mse_loss(x_hat, x)

    # delta loss
    dx_hat = temporal_diff(x_hat)
    dx = temporal_diff(x)

    loss_delta = F.mse_loss(dx_hat, dx)

    return alpha * loss_static + (1 - alpha) * loss_delta, loss_static, loss_delta

In [15]:
BATCH_SIZE = 4

In [ ]:
log = {
    "batch_idx": [],
    "loss": [],
}
ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)

ac_criterion = nn.MSELoss()
ac_optimizer = torch.optim.Adam(ac_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

ac_model.train()

batch  = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    mfccs = get_mfccs(ac_f, start, end).T

    if mfccs is None:
        continue

    batch.append(
        torch.tensor(mfccs, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1
    ac_batch = torch.stack(batch).to(device)  # (B,T,F)
    batch.clear()

    # ===== Acoustic =====
    ac_out, ac_z = ac_model(ac_batch)
    ac_loss, ac_static, ac_delta = combined_loss(
        ac_out, ac_batch
    )

    ac_optimizer.zero_grad()
    ac_loss.backward()
    ac_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(ac_loss.item())
    pbar.set_description(
        f"{batch_idx}: AC {ac_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if batch_idx % 1000 == 999:
        with torch.no_grad():
            ac_stats  = latent_stats(ac_z)

        print(
            f"[AC ] latent var mean={ac_stats['latent_var_mean']:.4e}, "
            f"min={ac_stats['latent_var_min']:.4e}"
        )
        print(
            f"[AC ] input std={batch_diversity(ac_batch):.4e}, "
            f"recon std={batch_diversity(ac_out):.4e}"
        )

        torch.save(ac_model.state_dict(), "ckpts/beus/ac_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/beus/ac_model_training_log.csv", index=False)

torch.save(ac_model.state_dict(), "ckpts/beus/ac_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/beus/ac_model_training_log.csv", index=False)

999: AC 314.9092:   0%|          | 999/226488 [00:39<2:13:28, 28.16it/s] 

[AC ] latent var mean=5.9466e+02, min=9.0358e-05
[AC ] input std=1.5277e+01, recon std=3.7728e+00


1999: AC 170.4559:   1%|          | 1996/226488 [01:14<2:11:50, 28.38it/s]

[AC ] latent var mean=1.3853e+02, min=2.5009e-03
[AC ] input std=1.1129e+01, recon std=3.1425e+00


2999: AC 160.0513:   1%|▏         | 3000/226488 [01:49<3:42:59, 16.70it/s]

...

96999: AC 389.7241:  43%|████▎     | 96998/226488 [51:30<33:46, 63.91it/s]  

[AC ] latent var mean=6.2827e+07, min=1.6292e+03
[AC ] input std=1.7816e+01, recon std=1.1504e+01


97999: AC 363.8606:  43%|████▎     | 97992/226488 [51:48<43:02, 49.76it/s]  

[AC ] latent var mean=3.5837e+07, min=2.0419e+01
[AC ] input std=1.5395e+01, recon std=8.2339e+00


98495: AC 184.0213:  43%|████▎     | 98492/226488 [51:57<33:59, 62.77it/s]  